# 06 · From a route to an observed journey

**How is a requested route different from a route inferred from GPS observations?**

This extends notebook 05 without changing it. We reuse Sanbapolis–BUC and the Trento OSM extract, then investigate the centre-of-Trento GPX from the original course material.

Goals: read a PBF source; start a routing service; inspect an HTTP response; distinguish routing from map matching; compare recorded points with an inferred network path; change one matching assumption.

Tools: GraphHopper 11.1, requests, GeoPandas and gpxpy. All maps use HOT OSM.

The local service runs in the same runtime as Python, including Colab. `localhost` here is that runtime, not the instructor's laptop. No temporary tunnel URL or API key is embedded. The first run downloads the Java application and imports the PBF. A ready-to-use source bundle is optional; all source URLs are visible.

### Source and credit

[OSM Estratti](https://osmit-estratti.wmcloud.org/): open the site, find the **Trento municipality**, inspect the coverage and download its PBF before running the next cell. The cell repeats the same download.

OSM data: © OpenStreetMap contributors, [ODbL](https://www.openstreetmap.org/copyright). Extract distributed by Wikimedia Italia. GPX teaching example: Maurizio Napolitano, linked original Gist. Snapshot dates differ; a match is not proof of historical road conditions.

[GraphHopper 11.x configuration](https://github.com/graphhopper/graphhopper/blob/11.x/config-example.yml) · [Map matching implementation](https://github.com/graphhopper/graphhopper/blob/11.x/web-bundle/src/main/java/com/graphhopper/resources/MapMatchingResource.java).

In [13]:
%pip install -q requests geopandas folium mapclassify matplotlib gpxpy pyyaml

In [14]:
from pathlib import Path
import json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from IPython.display import display
DATA = Path("data")
DATA.mkdir(exist_ok=True)

import gpxpy
from shapely.geometry import LineString
import subprocess, time, shutil

## 1. Download the original OSM data

What territory does this file cover? It contains OSM objects, not a ready-made route.

**Colab AI prompt:** Download the linked Trento PBF into DATA. Preserve the original bytes; show the URL and file size.

In [15]:
pbf_url = "https://osmit-estratti.wmcloud.org/output/pbf/comuni/022205_Trento.osm.pbf"
pbf_path = DATA / "Trento.osm.pbf"
print(pbf_url)
if not pbf_path.exists():
    response = requests.get(pbf_url, timeout=60)
    response.raise_for_status()
    pbf_path.write_bytes(response.content)
print("PBF bytes:", pbf_path.stat().st_size)

https://osmit-estratti.wmcloud.org/output/pbf/comuni/022205_Trento.osm.pbf
PBF bytes: 5793846


## 2. Prepare a routing service

GraphHopper imports OSM objects according to a profile. What changes when the travel mode changes?

**Colab AI prompt:** Download GraphHopper 11.1 and write an explicit foot-profile YAML. Keep profile preparation separate from route requests.

Run the following setup once. It uses **Java 17**. Check `java -version` if startup fails. In a Colab runtime without Java, install it in a separate cell with `!apt-get update -qq` followed by `!apt-get install -y -qq openjdk-17-jre-headless`.

The built-in `foot.json` profile selects and weights pedestrian edges. Its time estimates and preferences differ from the uniform 4 km/h exercise in 05. No elevation provider is configured. We disable speed-up preparations for this small teaching extract and retain normal routing and matching.

In [16]:
jar_url = "https://repo1.maven.org/maven2/com/graphhopper/graphhopper-web/11.1/graphhopper-web-11.1.jar"
jar_path = DATA / "graphhopper-11.1.jar"
if not jar_path.exists():
    response = requests.get(jar_url, timeout=90)
    response.raise_for_status()
    jar_path.write_bytes(response.content)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
config = """graphhopper:
  datareader.file: Trento.osm.pbf
  graph.location: graph-cache-foot-11.1
  import.osm.ignored_highways: ""
  graph.encoded_values: foot_access, foot_priority, foot_average_speed, foot_road_access, hike_rating, mtb_rating, country, road_class
  profiles:
    - name: foot
      custom_model_files: [foot.json]
  profiles_ch: []
  profiles_lm: []
server:
  application_connectors:
    - type: http
      port: 8989
      bind_host: 127.0.0.1
  admin_connectors:
    - type: http
      port: 8990
      bind_host: 127.0.0.1
"""
(DATA / "graphhopper.yml").write_text(config)
print(config)

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)

graphhopper:
  datareader.file: Trento.osm.pbf
  graph.location: graph-cache-foot-11.1
  import.osm.ignored_highways: ""
  graph.encoded_values: foot_access, foot_priority, foot_average_speed, foot_road_access, hike_rating, mtb_rating, country, road_class
  profiles:
    - name: foot
      custom_model_files: [foot.json]
  profiles_ch: []
  profiles_lm: []
server:
  application_connectors:
    - type: http
      port: 8989
      bind_host: 127.0.0.1
  admin_connectors:
    - type: http
      port: 8990
      bind_host: 127.0.0.1



## 3. Start and inspect the service

The log shows source import and server startup. A route can be requested only once the service is ready.

**Colab AI prompt:** Start Java in DATA with subprocess.Popen, save logs, wait at most 60 seconds and inspect /info. Reuse an already running local service on rerun.

In [17]:
GH_URL = "http://127.0.0.1:8989"
try:
    service_ready = requests.get(GH_URL + "/info", timeout=2).ok
except requests.RequestException:
    service_ready = False
if not service_ready:
    service_log = open(DATA / "graphhopper.log", "w")
    gh_process = subprocess.Popen(
        ["java", "-Xmx1g", "-jar", "graphhopper-11.1.jar", "server", "graphhopper.yml"],
        cwd=DATA.resolve(), stdout=service_log, stderr=subprocess.STDOUT,
    )
    for attempt in range(60):
        if gh_process.poll() is not None:
            raise RuntimeError((DATA / "graphhopper.log").read_text()[-5000:])
        try:
            if requests.get(GH_URL + "/info", timeout=1).ok:
                service_ready = True
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    if not service_ready:
        raise RuntimeError("Service not ready: inspect data/graphhopper.log before retrying.")
info = requests.get(GH_URL + "/info", timeout=10).json()
display(info)

{'bbox': [11.004763, 45.943473, 11.199303, 46.151855],
 'profiles': [{'name': 'foot'}],
 'version': '11.1',
 'elevation': False,
 'encoded_values': {'foot_access': ['true', 'false'],
  'foot_priority': ['>number', '<number'],
  'foot_average_speed': ['>number', '<number'],
  'foot_road_access': ['MISSING',
   'YES',
   'DESIGNATED',
   'DESTINATION',
   'PRIVATE',
   'USE_SIDEPATH',
   'NO'],
  'hike_rating': ['>number', '<number'],
  'mtb_rating': ['>number', '<number'],
  'country': ['MISSING',
   'AFG',
   'AGO',
   'AIA',
   'ALB',
   'AND',
   'ARE',
   'ARG',
   'ARM',
   'ATG',
   'AUS',
   'AUT',
   'AZE',
   'BDI',
   'BEL',
   'BEN',
   'BFA',
   'BGD',
   'BGR',
   'BHR',
   'BHS',
   'BIH',
   'BLR',
   'BLZ',
   'BMU',
   'BOL',
   'BRA',
   'BRB',
   'BRN',
   'BTN',
   'BWA',
   'CAF',
   'CAN',
   'CHE',
   'CHL',
   'CHN',
   'CIV',
   'CMR',
   'COD',
   'COG',
   'COK',
   'COL',
   'COM',
   'CPV',
   'CRI',
   'CUB',
   'CYM',
   'CYP',
   'CZE',
   'DEU',
   'DJI'

## 4. Ask for Sanbapolis–BUC

We request a possible walking route. Note that GraphHopper point parameters use latitude,longitude.

**Colab AI prompt:** Build a /route GET with two point parameters and points_encoded=false. Print the complete URL and inspect the response keys.

In [18]:
route_params = [("profile", "foot"), ("point", "46.04717,11.13261"),
                ("point", "46.05968,11.11552"), ("points_encoded", "false")]
response = requests.get(GH_URL + "/route", params=route_params, timeout=30)
print(response.url)
response.raise_for_status()
route_data = response.json()
(DATA / "route_foot.json").write_text(json.dumps(route_data, indent=2))
print(route_data.keys())
print(route_data["paths"][0].keys())

http://127.0.0.1:8989/route?profile=foot&point=46.04717%2C11.13261&point=46.05968%2C11.11552&points_encoded=false
dict_keys(['hints', 'info', 'paths'])
dict_keys(['distance', 'weight', 'time', 'transfers', 'legs', 'points_encoded', 'bbox', 'points', 'instructions', 'details', 'ascend', 'descend', 'snapped_waypoints'])


## 5. Read units and geometry

Distance is in metres; time is in milliseconds. With points_encoded=false, the coordinate array is longitude,latitude.

**Colab AI prompt:** Inspect distance, time and the first coordinates before making a LineString. Map the response on HOT OSM.

In [19]:
route_path = route_data["paths"][0]
print("Metres:", route_path["distance"])
print("Minutes:", route_path["time"] / 60000)
print("First coordinates:", route_path["points"]["coordinates"][:3])
route_gdf = gpd.GeoDataFrame({"model": ["GraphHopper foot"]},
    geometry=[LineString(route_path["points"]["coordinates"])], crs=4326)
display(route_gdf.explore(tiles="OpenStreetMap.HOT", color="blue"))

Metres: 2508.664
Minutes: 30.104066666666668
First coordinates: [[11.132612, 46.047169], [11.132622, 46.047181], [11.132525, 46.047296]]


## 6. Read the recorded journey

Now the question changes: which network path is compatible with these observations?

**Colab AI prompt:** Download the original journey_trento_downtown.gpx and inspect tracks, segments and its first point. Preserve the original file.

In [20]:
gpx_url = "https://gist.githubusercontent.com/napo/fc93af265b3d4a00e30be0bfc0e7ec06/raw/9fd6bc2a48538a3d4fad8789dd0315ca7f81fa8a/journey_trento_downtown.gpx"
gpx_path = DATA / "journey_trento_downtown.gpx"
if not gpx_path.exists():
    response = requests.get(gpx_url, timeout=40)
    response.raise_for_status()
    gpx_path.write_bytes(response.content)
gpx_text = gpx_path.read_text()
print(gpx_text[:600])
gpx = gpxpy.parse(gpx_text)
print("Tracks:", len(gpx.tracks))
print("Segments:", [len(track.segments) for track in gpx.tracks])
segment = gpx.tracks[0].segments[0]
print("First observation:", segment.points[0])

<?xml version="1.0"?>
<gpx xmlns:wptx1="http://www.garmin.com/xmlschemas/WaypointExtension/v1" xmlns:gpxedw="http://www.gpxeditor.co.uk/xmlschemas/WaypointExtension/v1" xmlns:gpxedts="http://www.gpxeditor.co.uk/xmlschemas/TrackSegExtension/v1" xmlns:gpxx="http://www.garmin.com/xmlschemas/GpxExtensions/v3" creator="GPXEditor" version="1.1" xmlns="http://www.topografix.com/GPX/1/1">
  <trk>
    <name>Journey</name>
    <trkseg>
      <trkpt lat="46.06676" lon="11.11954" />
      <trkpt lat="46.06677" lon="11.12024" />
    </trkseg>
    <trkseg>
      <trkpt lat="46.06677" lon="11.12024" />
     
Tracks: 1
Segments: [29]
First observation: [trkpt:46.06676,11.11954@None@None]


## 7. Map the observations first

This example has positions but may have no timestamps. A drawn line only joins consecutive observations.

**Colab AI prompt:** Create a point GeoDataFrame with sequence number and time. Plot points and the connecting line on HOT OSM. Do not invent missing timestamps.

In [21]:
rows = []
for sequence, point in enumerate(segment.points):
    rows.append({"sequence": sequence, "longitude": point.longitude,
                 "latitude": point.latitude, "time": point.time})
observations = gpd.GeoDataFrame(rows, geometry=gpd.points_from_xy(
    [row["longitude"] for row in rows], [row["latitude"] for row in rows]), crs=4326)
display(observations.head())
print("Missing timestamps:", observations["time"].isna().sum())
observed_line = gpd.GeoSeries([LineString(observations.geometry.tolist())], crs=4326)
m = observed_line.explore(tiles="OpenStreetMap.HOT", color="orange", name="Observed sequence")
observations.explore(m=m, color="orange", name="Recorded points", tooltip=["sequence"], marker_kwds={"radius": 4})
folium.LayerControl().add_to(m)
display(m)

,sequence,longitude,latitude,time,geometry
0,0,11.11954,46.06676,None,POINT (11.11954 46.06676)
1,1,11.12024,46.06677,None,POINT (11.12024 46.06677)


Missing timestamps: 2


## 8. Match the observations to the network

gps_accuracy is an assumed observation-error scale in metres, not a measured accuracy for every point.

**Colab AI prompt:** POST the unchanged GPX to /match with foot, gps_accuracy=10 and points_encoded=false. Inspect response keys and map_matching statistics.

In [22]:
match_params = {"profile": "foot", "gps_accuracy": 10,
                "points_encoded": "false", "type": "json"}
response = requests.post(GH_URL + "/match", params=match_params,
    data=gpx_path.read_bytes(), headers={"Content-Type": "application/gpx+xml"}, timeout=60)
response.raise_for_status()
match_data = response.json()
(DATA / "match_10.json").write_text(json.dumps(match_data, indent=2))
print(match_data.keys())
display(match_data.get("map_matching", {}))
matched = gpd.GeoDataFrame({"gps_accuracy_m": [10]},
    geometry=[LineString(match_data["paths"][0]["points"]["coordinates"])], crs=4326)

dict_keys(['hints', 'info', 'paths', 'map_matching'])


{'original_distance': 1342.374625154211,
 'distance': 1367.4648522055522,
 'time': 984578}

## 9. Compare before and after

A matched line is a model result, not an independent observation.

**Colab AI prompt:** Overlay orange recorded points and the blue matched path on HOT OSM. Inspect one ambiguous crossing.

In [23]:
m = matched.explore(tiles="OpenStreetMap.HOT", color="blue", name="Matched path", style_kwds={"weight": 5})
observed_line.explore(m=m, color="orange", name="Recorded sequence", style_kwds={"dashArray": "5,5"})
observations.explore(m=m, color="orange", name="Recorded points", tooltip=["sequence"], marker_kwds={"radius": 4})
folium.LayerControl(collapsed=False).add_to(m)
display(m)

## 10. Change one assumption

Repeat the same match with a 30-metre error scale. The model may or may not choose a different path.

**Colab AI prompt:** Reuse the same GPX and profile, change only gps_accuracy to 30, compare statistics and map both results.

In [24]:
match_params_30 = {**match_params, "gps_accuracy": 30}
response = requests.post(GH_URL + "/match", params=match_params_30,
    data=gpx_path.read_bytes(), headers={"Content-Type": "application/gpx+xml"}, timeout=60)
response.raise_for_status()
match_data_30 = response.json()
(DATA / "match_30.json").write_text(json.dumps(match_data_30, indent=2))
matched_30 = gpd.GeoDataFrame({"gps_accuracy_m": [30]},
    geometry=[LineString(match_data_30["paths"][0]["points"]["coordinates"])], crs=4326)
comparison = pd.DataFrame([
    {"accuracy_assumption_m": 10, **match_data.get("map_matching", {})},
    {"accuracy_assumption_m": 30, **match_data_30.get("map_matching", {})},
])
display(comparison)
m = matched.explore(tiles="OpenStreetMap.HOT", color="blue", name="10 m assumption")
matched_30.explore(m=m, color="red", name="30 m assumption")
observations.explore(m=m, color="orange", name="Observations", tooltip=["sequence"], marker_kwds={"radius": 3})
folium.LayerControl(collapsed=False).add_to(m)
display(m)

,accuracy_assumption_m,original_distance,distance,time
0,10,1342.374625,1367.464852,984578
1,30,1342.374625,1358.954817,978447


## Explain one ambiguous result

Which parts are measured positions, which are connecting lines and which are inferred paths? What evidence supports the chosen street at one crossing? What happens if the network omits the real path? Can this position-only GPX establish speed or a stop duration?

Keep original points separate from matched geometry. Do not transfer GraphHopper's estimated travel time to the observation timestamps. In notebook 07 we use the original timestamped trip to examine movement and the instructor's incident case.

**Optional cleanup:** stop only the local process created by this notebook, after finishing the service exercises:
```python
if "gh_process" in globals() and gh_process.poll() is None:
    gh_process.terminate()
```